# VIPER — Video Identity Perturbation and Extraction Residual

**Deepfake Detection via Biometric Identity Consistency Analysis**

[![GitHub](https://img.shields.io/badge/GitHub-rxbinsingh/VIPER-blue?logo=github)](https://github.com/rxbinsingh/VIPER)

---

**Author:** Robin Singh · Bennett University · 2025

**Architecture:** CLIP ViT-L/14 (frozen) + Identity-Anchored Preprocessing + Fusion MLP

**Results:** AUC 0.9909 | Accuracy 95.2% | Inference ~4s/video

---

### Pipeline

```
Video → Face Detection (InsightFace) → 16 Face Crops (224×224)
                                          │
      ┌────────────────────────────────────┤
      │                                    │
      ▼                                    ▼
  Identity Anchor                    CLIP ViT-L/14
  (ArcFace + DCT + dlib BCR)         (frozen, 768-dim)
      │                                    │
      ▼                                    ▼
  16-dim hand features              768-dim video embedding
      │                                    │
      └──────────────┬─────────────────────┘
                     ▼
              Fusion MLP (784 → 512 → 128 → 1)
                     │
                REAL / FAKE
```

### Before Running

Upload `dataset_production/` to Google Drive at `MyDrive/VIPER/dataset_production/`

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1 — Environment Setup
# ═══════════════════════════════════════════════════════════════

import torch
assert torch.cuda.is_available(), 'GPU required. Go to Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!pip install -q open_clip_torch insightface onnxruntime-gpu dlib
!pip install -q opencv-python-headless scipy scikit-learn tqdm matplotlib gradio
print('\n✓ All dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2 — Mount Drive + Clone Repository
# ═══════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob
from pathlib import Path

if not os.path.exists('/content/VIPER'):
    !git clone https://github.com/rxbinsingh/VIPER /content/VIPER
else:
    !git -C /content/VIPER pull --quiet

os.chdir('/content/VIPER')
sys.path.insert(0, '/content/VIPER')

# Paths
DATA_DIR  = '/content/drive/MyDrive/VIPER/dataset_production'
CACHE_DIR = '/content/drive/MyDrive/VIPER/cache'
SAVE_DIR  = '/content/drive/MyDrive/VIPER/checkpoints'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# Verify dataset
assert os.path.exists(DATA_DIR), f'Upload dataset_production/ to Google Drive at MyDrive/VIPER/'
print(f'Dataset:  {DATA_DIR}')
print(f'Cache:    {CACHE_DIR}')
print(f'Saves:    {SAVE_DIR}\n')

for folder in ['real', 'face_swap', 'expression_swap', 'fullbody_gan']:
    n = len(glob.glob(f'{DATA_DIR}/{folder}/*.mp4'))
    print(f'  {folder:<20}: {n} videos')

cached = len(glob.glob(f'{CACHE_DIR}/*.npz'))
print(f'\nCache: {cached} videos already preprocessed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3 — Load Dataset Splits
# ═══════════════════════════════════════════════════════════════

import numpy as np
from src.dataset import load_samples, print_split_stats

train_samples = load_samples(DATA_DIR, split='train', seed=42)
val_samples   = load_samples(DATA_DIR, split='val',   seed=42)
test_samples  = load_samples(DATA_DIR, split='test',  seed=42)

print_split_stats(train_samples, 'Train')
print_split_stats(val_samples,   'Val')
print_split_stats(test_samples,  'Test')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4 — Preprocess Videos (cached to Drive, ~1.5 hours)
# ═══════════════════════════════════════════════════════════════
# If session dies, restart from Cell 1 — cached videos are skipped.

import cv2
from tqdm import tqdm
from src.preprocessing import preprocess_video
from src.anchor_extractor import build_identity_anchor
from src.displacement_probe import compute_all_residuals

# Patch BCR to return None (dlib BCR computed separately if needed)
import src.anchor_extractor as ae
ae.build_biomech_anchor = lambda frames: None

def process_and_cache(video_path, label):
    cp = f'{CACHE_DIR}/{Path(video_path).stem}.npz'
    if os.path.exists(cp):
        return 'cached'
    try:
        preprocessed = preprocess_video(video_path, num_frames=16, n_anchor=8)
        if not preprocessed['valid']:
            return 'no_face'
        anchor    = build_identity_anchor(preprocessed)
        residuals = compute_all_residuals(preprocessed, anchor)
        g, t = residuals['gir_stats'], residuals['tfr_stats']
        hand_feats = [g['MR'],g['TV'],g['DSR'], t['MR'],t['TV'],t['DSR'],
                      0.0,0.0,0.0,0.0, anchor['anchor_quality'],0.0,0.0,0.0,0.0,0.0]
        crops_rgb = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in preprocessed['video_frames']]
        np.savez_compressed(cp, crops=np.stack(crops_rgb),
                            hand_feats=np.array(hand_feats, dtype=np.float32),
                            label=np.array(label, dtype=np.int32))
        return 'ok'
    except Exception as e:
        return f'error: {e}'

all_samples = train_samples + val_samples + test_samples
counts = {'ok':0, 'cached':0, 'no_face':0, 'error':0}
for video_path, label, _ in tqdm(all_samples, desc='Preprocessing'):
    r = process_and_cache(video_path, label)
    counts['cached' if r=='cached' else 'ok' if r=='ok' else 'no_face' if r=='no_face' else 'error'] += 1

total_ok = counts['ok'] + counts['cached']
print(f'\nDone: {total_ok}/{len(all_samples)} usable ({total_ok/len(all_samples)*100:.0f}%)')
print(f'  New: {counts["ok"]} | Cached: {counts["cached"]} | No face: {counts["no_face"]} | Errors: {counts["error"]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5 — Build Datasets + CLIP Model
# ═══════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from PIL import Image
import open_clip

device = 'cuda'

# ── CLIP preprocessing ────────────────────────────────────────
CLIP_TF = T.Compose([
    T.Resize(224), T.CenterCrop(224),
    T.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                std=[0.26862954,0.26130258,0.27577711]),
])

class VIPERDataset(Dataset):
    def __init__(self, samples, cache_dir, num_frames=16, train=True):
        self.samples = [(p,l) for p,l,_ in samples
                        if os.path.exists(f'{cache_dir}/{Path(p).stem}.npz')]
        self.cache_dir, self.num_frames = cache_dir, num_frames
        self.base = T.Compose([T.RandomHorizontalFlip(0.5 if train else 0), T.ToTensor()])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        data = np.load(f'{self.cache_dir}/{Path(path).stem}.npz')
        crops, hand = data['crops'], data['hand_feats']
        tensors = [CLIP_TF(self.base(Image.fromarray(crops[i]))) for i in range(min(len(crops),self.num_frames))]
        while len(tensors) < self.num_frames: tensors.append(tensors[-1])
        return {'crops': torch.stack(tensors[:self.num_frames]),
                'hand_feats': torch.tensor(hand, dtype=torch.float32),
                'label': torch.tensor(label, dtype=torch.float32)}

train_ds = VIPERDataset(train_samples, CACHE_DIR, train=True)
val_ds   = VIPERDataset(val_samples, CACHE_DIR, train=False)
test_ds  = VIPERDataset(test_samples, CACHE_DIR, train=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f'Datasets: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')

# ── CLIP ViT-L/14 + Fusion MLP ───────────────────────────────
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
clip_model = clip_model.to(device).eval()
for p in clip_model.parameters(): p.requires_grad = False

class VIPERv3(nn.Module):
    def __init__(self, clip_visual, dropout=0.4):
        super().__init__()
        self.clip = clip_visual
        self.head = nn.Sequential(
            nn.Linear(784, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout*0.5),
            nn.Linear(128, 1))
        nn.init.xavier_uniform_(self.head[-1].weight, gain=0.1)
        nn.init.zeros_(self.head[-1].bias)
    def forward(self, crops, hand):
        B, T_, C, H, W = crops.shape
        with torch.no_grad():
            embs = self.clip(crops.view(B*T_, C, H, W))
        embs = embs.view(B, T_, -1).mean(dim=1)
        return self.head(torch.cat([embs.float(), hand], dim=1)).squeeze(-1)

model = VIPERv3(clip_model.visual, dropout=0.4).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: VIPERv3 (CLIP ViT-L/14 frozen + MLP)')
print(f'Trainable parameters: {trainable:,}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 6 — Training (~25 minutes)
# ═══════════════════════════════════════════════════════════════

EPOCHS = 15
n_real = sum(1 for _,l,_ in train_samples if l==0)
n_fake = sum(1 for _,l,_ in train_samples if l==1)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_real/n_fake]).to(device))
optimizer = AdamW(model.head.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    all_l, all_p = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False, desc='Train' if train else 'Val'):
            crops, hand, labels = batch['crops'].to(device), batch['hand_feats'].to(device), batch['label'].to(device)
            if train: optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits = model(crops, hand)
                loss = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            all_l.extend(labels.cpu().numpy())
            all_p.extend(torch.sigmoid(logits).detach().cpu().numpy())
    return roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0.0

best_auc = 0.0
print(f'Training VIPERv3 for {EPOCHS} epochs...\n')
for epoch in range(1, EPOCHS+1):
    tr_auc = run_epoch(train_loader, train=True)
    vl_auc = run_epoch(val_loader, train=False)
    scheduler.step()
    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), f'{SAVE_DIR}/viper_best_v3.pt')
        flag = '  ← best'
    print(f'Epoch {epoch:02d}/{EPOCHS}  Train={tr_auc:.4f}  Val={vl_auc:.4f}{flag}')

print(f'\nBest Val AUC: {best_auc:.4f}')
print(f'Checkpoint: {SAVE_DIR}/viper_best_v3.pt')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 7 — Test Evaluation + TTA
# ═══════════════════════════════════════════════════════════════

model.load_state_dict(torch.load(f'{SAVE_DIR}/viper_best_v3.pt', map_location=device))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test + TTA'):
        crops, hand = batch['crops'].to(device), batch['hand_feats'].to(device)
        with torch.amp.autocast('cuda'):
            l1 = model(crops, hand)
            l2 = model(torch.flip(crops, [-1]), hand)  # horizontal flip TTA
        all_labels.extend(batch['label'].numpy())
        all_probs.extend(torch.sigmoid((l1+l2)/2).cpu().numpy())

auc = roc_auc_score(all_labels, all_probs)
preds = [1 if p>0.5 else 0 for p in all_probs]
acc = accuracy_score(all_labels, preds)
cm = confusion_matrix(all_labels, preds)

print(f'\n{"═"*55}')
print(f'  VIPER v3 — FINAL TEST RESULTS')
print(f'{"═"*55}')
print(f'  AUC-ROC:  {auc:.4f}')
print(f'  Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'\n  Confusion Matrix:')
print(f'    TN={cm[0,0]}  FP={cm[0,1]}')
print(f'    FN={cm[1,0]}  TP={cm[1,1]}')
print(f'\n{classification_report(all_labels, preds, target_names=["Real","Fake"])}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 8 — Per-Fake-Type Breakdown
# ═══════════════════════════════════════════════════════════════

type_labels = {'face_swap':[], 'expression_swap':[], 'fullbody_gan':[], 'real':[]}
type_probs  = {'face_swap':[], 'expression_swap':[], 'fullbody_gan':[], 'real':[]}

for video_path, label, label_str in test_samples:
    cp = f'{CACHE_DIR}/{Path(video_path).stem}.npz'
    if not os.path.exists(cp): continue
    data = np.load(cp)
    crops, hand = data['crops'], data['hand_feats']
    T_actual = min(len(crops), 16)
    base_tf = T.Compose([T.ToTensor()])
    tensors = [CLIP_TF(base_tf(Image.fromarray(crops[i]))) for i in range(T_actual)]
    while len(tensors) < 16: tensors.append(tensors[-1])
    crops_t = torch.stack(tensors[:16]).unsqueeze(0).to(device)
    hand_t = torch.tensor(hand, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            l1 = model(crops_t, hand_t)
            l2 = model(torch.flip(crops_t, [-1]), hand_t)
        prob = torch.sigmoid((l1+l2)/2).item()
    type_labels[label_str].append(label)
    type_probs[label_str].append(prob)

print(f'{"Type":<20} {"AUC":>8} {"Acc":>8} {"N":>5}')
print('─'*45)
for ft in ['face_swap', 'expression_swap', 'fullbody_gan']:
    if not type_labels[ft]: print(f'{ft:<20} {"N/A":>8} {"N/A":>8} {0:>5}'); continue
    cl = type_labels['real'] + type_labels[ft]
    cp_ = type_probs['real'] + type_probs[ft]
    a = roc_auc_score(cl, cp_)
    p = [1 if x>0.5 else 0 for x in cp_]
    print(f'{ft:<20} {a:>8.4f} {accuracy_score(cl,p):>8.4f} {len(type_labels[ft]):>5}')
print('─'*45)
print(f'{"ALL":<20} {auc:>8.4f} {acc:>8.4f} {len(all_labels):>5}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 9 — Save Results + Launch Demo
# ═══════════════════════════════════════════════════════════════

import json

results = {
    'model': 'VIPER v3 (CLIP ViT-L/14 + TTA)',
    'test_auc': round(auc, 4),
    'test_accuracy': round(acc, 4),
    'val_auc': round(best_auc, 4),
    'confusion_matrix': cm.tolist(),
    'backbone': 'CLIP ViT-L/14 (frozen)',
    'classifier': 'MLP 784→512→128→1',
    'training_time': '~25 minutes on T4',
    'inference_speed': '~4s per video (end-to-end)',
}
with open(f'{SAVE_DIR}/results_v3.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to Drive.')
print(f'\nFinal: AUC={auc:.4f} | Accuracy={acc*100:.1f}% | Val AUC={best_auc:.4f}')
print(f'\nTo launch Gradio demo:')
print(f'  !cp {SAVE_DIR}/viper_best_v3.pt /content/VIPER/checkpoints/viper_best.pt')
print(f'  !python /content/VIPER/app.py')